In [1]:
# ===========================================
# Libraries
# ===========================================

import sys
import numpy as np
import emcee

sys.path.insert(0, "/home/brunowesley/projetos/MCMC-cosmo/Codes/Model-comparison/Bayesian-Evidence/MCEvidence")
from MCEvidence import MCEvidence

In [2]:
# ===========================================
# Load chains (Baseline and Baseline + FRB)
# ===========================================

ROOT = "/home/brunowesley/projetos/MCMC-cosmo/Codes/GR-based-models/wCDM"

datasets = {
    "Baseline": {
        "samples": f"{ROOT}/CC_SNe_BAO/flat_samples_wCDM_cc_sne_bao.npy",
        "logprob": f"{ROOT}/CC_SNe_BAO/log_prob_wCDM_cc_sne_bao.npy",
        "nwalkers": 40,
    },
    "Baseline + FRB": {
        "samples": f"{ROOT}/CC_SNe_BAO_FRB/flat_samples_wCDM_cc_sne_bao_frb.npy",
        "logprob": f"{ROOT}/CC_SNe_BAO_FRB/log_prob_wCDM_cc_sne_bao_frb.npy",
        "nwalkers": 56,
    },
}

for tag, d in datasets.items():
    d["flat_samples"] = np.load(d["samples"])
    d["logprob_arr"]  = np.load(d["logprob"])
    d["ndim"] = d["flat_samples"].shape[1]
    ns = d['flat_samples'].shape
    print(f"{tag:16s}: samples={ns}, ndim={d['ndim']}, nwalkers={d['nwalkers']}")

Baseline        : samples=(1192280, 5), ndim=5, nwalkers=40
Baseline + FRB  : samples=(1665944, 7), ndim=7, nwalkers=56


In [3]:
# ===========================================
# Prior volumes (from Table 1 of the manuscript)
# ===========================================
# wCDM Baseline params: H0, Om0, Ob, MB, w
# wCDM Baseline+FRB params: H0, Om0, Ob, MB, w, sigma_host, exp_mu

bounds = {
    "H0":         (50, 90),
    "Om0":        (0.1, 0.6),
    "Ob":         (0.01, 0.1),
    "MB":         (-20.0, -18.0),
    "w":          (-3, 1),
    "w0":         (-3, 1),
    "wa":         (-3, 2),
    "b":          (1.6, 12),
    "mu_HS":      (0, 300),
    "lambda_S":   (1.54, 10),
    "sigma_host": (0.2, 2.0),
    "exp_mu":     (20, 200),
}

def prior_volume(param_names):
    V = 1.0
    for p in param_names:
        lo, hi = bounds[p]
        V *= (hi - lo)
    return V

datasets["Baseline"]["params"] = ['H0', 'Om0', 'Ob', 'MB', 'w']
datasets["Baseline + FRB"]["params"] = ['H0', 'Om0', 'Ob', 'MB', 'w', 'sigma_host', 'exp_mu']

for tag, d in datasets.items():
    d["priorvolume"] = prior_volume(d["params"])
    pv = d['priorvolume']
    print(f"{tag:16s}: params={d['params']}, prior volume={pv:.6g}")

Baseline        : params=['H0', 'Om0', 'Ob', 'MB', 'w'], prior volume=14.4
Baseline + FRB  : params=['H0', 'Om0', 'Ob', 'MB', 'w', 'sigma_host', 'exp_mu'], prior volume=4665.6


In [4]:
# ===========================================
# Thinning: recover (nsteps, nwalkers, ndim) shape and thin by tau_max/2
# ===========================================
# emcee's get_chain(flat=True) flattens in (step, walker) order, so the
# original 3D shape is exactly reconstructible from the flat array given
# nwalkers. The Heavens et al. (2017) method assumes independent samples;
# raw MCMC chains are autocorrelated, so we thin by tau_max/2 (the same
# criterion already used in the convergence diagnostics of the original
# fitting notebooks) before estimating the evidence.

def thin_chain(flat_samples, logprob, nwalkers):
    ndim = flat_samples.shape[1]
    nsteps_post = len(flat_samples) // nwalkers

    chain3d = flat_samples.reshape(nsteps_post, nwalkers, ndim)
    lp3d    = logprob.reshape(nsteps_post, nwalkers)

    tau = emcee.autocorr.integrated_time(chain3d, tol=0)
    tau_max = np.max(tau)
    nthin = max(1, int(tau_max / 2))

    thin_samples = chain3d[::nthin].reshape(-1, ndim)
    thin_logprob = lp3d[::nthin].reshape(-1)

    return thin_samples, thin_logprob, tau_max, nthin

for tag, d in datasets.items():
    s, lp, tau_max, nthin = thin_chain(d["flat_samples"], d["logprob_arr"], d["nwalkers"])
    d["thin_samples"] = s
    d["thin_logprob"] = lp
    d["tau_max"] = tau_max
    d["nthin"] = nthin
    N_raw = len(d['flat_samples'])
    N_thin = len(s)
    print(f"{tag:16s}: tau_max={tau_max:.2f}, nthin={nthin}, N_raw={N_raw}, N_thin={N_thin}")

Baseline        : tau_max=63.75, nthin=31, N_raw=1192280, N_thin=38480


Baseline + FRB  : tau_max=83.74, nthin=41, N_raw=1665944, N_thin=40656


In [5]:
# ===========================================
# Evidence estimation (Heavens et al. 2017, k=1 nearest-neighbour)
# ===========================================

def evidence_estimate(samples, logprob, ndim, priorvolume, k=1):
    weight = np.ones(len(logprob))
    chain = np.column_stack([weight, -logprob, samples])
    mce = MCEvidence([chain], ischain=True, ndim=ndim, kmax=k + 1,
                      priorvolume=priorvolume, verbose=0)
    lnZ = mce.evidence()[0]
    N = len(logprob)
    sigma_lnZ = 1.0 / np.sqrt(N * k + 1)
    return lnZ, sigma_lnZ, N

print("\n" + "=" * 60)
print("             wCDM | Bayesian evidence (k=1 NN)              ")
print("=" * 60)

for tag, d in datasets.items():
    lnZ, sigma_lnZ, N = evidence_estimate(d["thin_samples"], d["thin_logprob"],
                                           d["ndim"], d["priorvolume"])
    d["lnZ"] = lnZ
    d["sigma_lnZ"] = sigma_lnZ
    d["N_used"] = N
    print(f"\n{tag}:")
    print(f"    ln Z = {lnZ:.3f} +/- {sigma_lnZ:.4f}  (N={N}, ndim={d['ndim']})")

print("\n" + "=" * 60)


             wCDM | Bayesian evidence (k=1 NN)              



Baseline:
    ln Z = -746.777 +/- 0.0051  (N=38480, ndim=5)



Baseline + FRB:
    ln Z = -1400.303 +/- 0.0050  (N=40656, ndim=7)



In [6]:
# ===========================================
# Save results (for the Bayes-factor summary notebook)
# ===========================================

np.savez("evidence_wCDM.npz",
         tags=list(datasets.keys()),
         lnZ=np.array([datasets[t]["lnZ"] for t in datasets]),
         sigma_lnZ=np.array([datasets[t]["sigma_lnZ"] for t in datasets]),
         ndim=np.array([datasets[t]["ndim"] for t in datasets]),
         N_used=np.array([datasets[t]["N_used"] for t in datasets]),
         priorvolume=np.array([datasets[t]["priorvolume"] for t in datasets]))

with open("evidence_wCDM.txt", "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("             wCDM | Bayesian evidence (k=1 NN)              \n")
    f.write("=" * 60 + "\n")
    for tag, d in datasets.items():
        f.write(f"\n{tag}:\n")
        f.write(f"    ln Z = {d['lnZ']:.3f} +/- {d['sigma_lnZ']:.4f}  "
                f"(N={d['N_used']}, ndim={d['ndim']}, prior volume={d['priorvolume']:.6g})\n")
    f.write("\n" + "=" * 60 + "\n")

print("Saved: evidence_wCDM.npz, evidence_wCDM.txt")

Saved: evidence_wCDM.npz, evidence_wCDM.txt
